# Function Tools

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

MODEL = 'gpt-4o-mini'

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    base_url=os.environ.get("OPENAI_API_BASE")
)

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant for an Airline called FlightAI. "
    "Give short, courteous answers, no more than 1 sentence. "
    "Always be accurate. If you don't know the answer, say so."
)

In [ ]:
# --- Data ---
CITY_TICKET_PRICES = {
    "london": "$799",
    "paris": "$999",
    "tokyo": "$1099",
}

# --- Function Definitions ---

def get_flight_ticket_price(destination_city: str) -> str:
    """Retrieves the ticket price for a given city.

    Args:
        destination_city: The city to get the price for.

    Returns:
        The ticket price as a string, or "Price Not Found" if the city is not in the database.
    """
    city_lower = destination_city.lower()
    return CITY_TICKET_PRICES.get(city_lower, "Price Not Found")

In [ ]:
get_price_tool_spec = {
    "name": "get_flight_ticket_price",
    "description": "Provides the price of a return flight ticket to the specified destination city. Use this whenever a user inquires about ticket prices.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the user wants to fly to.",
            },
        },
        "required": ["destination_city"],
    },
}

available_tools = [
    {"type": "function", "function": get_price_tool_spec},
]


In [ ]:
def execute_tool_call(tool_call):
    """Executes a tool call received from OpenAI.

    Args:
        tool_call: The tool call object from the OpenAI API response.

    Returns:
        A dictionary representing the tool's response, ready to be sent back to OpenAI.
    """
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    if function_name == "get_flight_ticket_price":
        city = arguments.get("destination_city")
        price = get_flight_ticket_price(city)
        tool_response = {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": json.dumps({"destination_city": city, "price": price}),
        }
        return tool_response
    else:
        raise ValueError(f"Unknown tool call: {function_name}")

In [ ]:
def run_conversation(user_message: str, chat_history: list) -> str:
    """Manages the conversation with the OpenAI API.

    Args:
        user_message: The user's current message.
        chat_history: A list of previous messages in the conversation.

    Returns:
        The chatbot's response to the user.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + chat_history
    messages.append({"role": "user", "content": user_message})

    # First API call
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=available_tools,
    )
    response_message = response.choices[0].message

    while response_message.tool_calls:
        tool_call = response_message.tool_calls[0]
        tool_response = execute_tool_call(tool_call)
        messages.append(response_message)
        messages.append(tool_response)

        # Second API call
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
        )
        response_message = response.choices[0].message

    return response_message.content

In [ ]:
gr.ChatInterface(fn=run_conversation, type="messages").launch()